# Lab 03 — Certified robustness

**Duration: 15–20 minutes.** An attack from Lab 02 searches for a misclassification. Here we ask whether **every input in a small neighborhood is classified correctly**.

Complete **two short code exercises**: call auto-LiRPA to compute bounds, then check whether the margin bounds certify the true label. Replace the three `...` expressions. All setup, model and perturbation code is provided. Run the cells in order and save your work locally.

This practical adapts the [auto-LiRPA quick-start tutorial](http://PaperCode.cc/AutoLiRPA-Demo), using its small pretrained ResNet-18 on one CIFAR-10 image.

## Setup

Run `uv sync --locked` in this lab folder and select its `.venv` notebook kernel. See [README.md](README.md). The lab runs on CPU; downloads are cached in `data/`.

In [ ]:
import sys
import warnings
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms
from IPython.display import display
from auto_LiRPA import BoundedModule, BoundedTensor, PerturbationLpNorm

from lab_utils import CIFAR_STD, load_demo_model

# Matrix mode is intentional for this tiny network. Hide its repeated size notices.
warnings.filterwarnings(
    "ignore", message=r"Creating an identity matrix with size .*",
    category=UserWarning, module=r"auto_LiRPA\.bound_general",
)
# Upstream sets a global warning filter; keep third-party deprecations out of the lab output.
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"matplotlib\._mathtext|pyparsing.*")

assert sys.version_info[:2] == (3, 11), "Select Lab 03's Python 3.11 .venv kernel."
SEED = 3
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}; torch: {torch.__version__}; auto-LiRPA: {version('auto-lirpa')}")

## Model and image — provided

We keep the tutorial's narrow ResNet-18. The model accepts raw pixels in **[0, 1]** and normalizes them internally, so epsilon has the same pixel-space meaning throughout the lab. The naturally trained checkpoint is usually certifiable only for small perturbations. Batch normalization stays in evaluation mode.

We examine one fixed image; this is not a test-set accuracy estimate.

In [ ]:
model = load_demo_model(device)
model.requires_grad_(False)  # Bounds are evaluated without training the model.
test_data = datasets.CIFAR10(
    "data", train=False, download=True, transform=transforms.ToTensor()
)
INDEX = 123
pixels, label = test_data[INDEX]
image = pixels.unsqueeze(0).to(device)
labels = torch.tensor([label], device=device)
classes = test_data.classes

with torch.no_grad():
    logits = model(image)
predicted = logits.argmax(dim=1)
clean_correct = predicted.eq(labels)
print(f"Image {INDEX}: true label = {classes[label]}, prediction = {classes[predicted.item()]}")
print(f"Clean classification correct: {clean_correct.item()}")
plt.figure(figsize=(3, 3))
plt.imshow(pixels.permute(1, 2, 0))
plt.title(f"True: {classes[label]} / predicted: {classes[predicted.item()]}")
plt.axis("off")
plt.show()

## Perturbations — provided

We allow $\|x'-x\|_\infty\leq\epsilon$ and valid pixels $0\leq x'\leq1$. An epsilon of $1/255$ is one intensity level on a 0–255 scale.

The provided code wraps the model and defines the clipped pixel box. Normalization stays inside the model, so epsilon is always measured on raw pixels.

In [ ]:
bounded_model = BoundedModule(
    model, torch.zeros_like(image), bound_opts={"conv_mode": "matrix"}, device=device,
)
bounded_model.eval()
with torch.no_grad():
    torch.testing.assert_close(bounded_model(image), logits, atol=1e-5, rtol=1e-4)

In [ ]:
def bounded_input(x, epsilon):
    x_L = (x - epsilon).clamp(0, 1)
    x_U = (x + epsilon).clamp(0, 1)
    perturbation = PerturbationLpNorm(norm=np.inf, eps=epsilon, x_L=x_L, x_U=x_U)
    return BoundedTensor(x, perturbation)

EPSILON = 0.1 / 255
bounded_image = bounded_input(image, EPSILON)
assert torch.all(bounded_image.ptb.x_L <= image) and torch.all(image <= bounded_image.ptb.x_U)
assert bounded_image.ptb.x_L.min() >= 0 and bounded_image.ptb.x_U.max() <= 1
print(f"Pixel-space epsilon = {EPSILON:.6f} ({EPSILON * 255:g}/255)")

## Exercise 1 — Compute the bounds (5–7 minutes)

**IBP** propagates intervals and is usually fast but loose. **CROWN** uses linear relaxations and often gives tighter bounds. Both return $\ell_j\leq f_j(x')\leq u_j$ for every allowed input.

Complete `result` with `bounded_model.compute_bounds`. Pass:

- `x=(bounded_input(x, epsilon),)` — the bounded input as a one-element tuple;
- `method=method` — select IBP or CROWN;
- `C=C` — the optional specification, used in Exercise 2.

Run the cell and compare the printed interval widths.

In [ ]:
def compute_bounds(x, epsilon, method, C=None):
    with torch.no_grad():
        result = ...  # TODO: call bounded_model.compute_bounds with x, method, and C.
    assert isinstance(result, tuple) and len(result) == 2, "Complete Exercise 1's bound computation."
    return result[0].detach(), result[1].detach()

METHODS = ("IBP", "CROWN")
logit_bounds = {}
logit_table = pd.DataFrame({"class": classes, "clean logit": logits[0].cpu().numpy()})
for method in METHODS:
    lb, ub = compute_bounds(image, EPSILON, method)
    assert torch.all(lb <= ub + 1e-5)
    assert torch.all(logits >= lb - 1e-4) and torch.all(logits <= ub + 1e-4)
    logit_bounds[method] = (lb, ub)
    logit_table[f"{method} lower"] = lb[0].cpu().numpy()
    logit_table[f"{method} upper"] = ub[0].cpu().numpy()
    print(f"{method}: mean interval width = {(ub - lb).mean().item():.4f}")
display(logit_table.round(4))

## Exercise 2 — Complete the certificate (5–7 minutes)

Correct classification requires the true class to beat all competitors: $f_y(x')-f_j(x')>0$ for every $j\ne y$. The provided matrix `C` asks auto-LiRPA to bound these nine margins directly.

Complete two expressions in the experiment:

1. `minimum_margin`: use `lb_margin.amin(dim=1)` to get the smallest margin lower bound for each image.
2. `certified`: combine `clean_correct` and `minimum_margin > CERT_TOL` with `&`.

Then run the cell to see how certification changes with epsilon. `CERT_TOL` is a small positive tolerance for floating-point bounds.

In [ ]:
# Specification matrix C is provided: each row represents true logit minus a competitor.
competitors = torch.arange(10, device=device)
competitors = competitors[competitors != labels.item()]
eye = torch.eye(10, device=device)
C = eye[labels][:, None, :] - eye[competitors][None, :, :]

In [ ]:
CERT_TOL = 1e-6
EPSILON_LEVELS = [0, 0.1, 0.5]  # Units of 1/255.
rows = []
for level in EPSILON_LEVELS:
    for method in METHODS:
        lb_margin, _ = compute_bounds(image, level / 255, method, C=C)
        minimum_margin = ...  # TODO: smallest margin lower bound per image.
        certified = ...  # TODO: clean-correct AND every margin certified positive.
        assert isinstance(minimum_margin, torch.Tensor) and isinstance(certified, torch.Tensor), "Complete Exercise 2's certificate."
        assert minimum_margin.shape == labels.shape and certified.dtype == torch.bool
        rows.append({
            "epsilon (1/255)": level, "method": method,
            "minimum margin lower bound": minimum_margin.item(),
            "certified correct": certified.item(),
        })
results = pd.DataFrame(rows)
display(results.round(4))

## Read the result

`True` certifies the correct label throughout the specified box. `False` is **inconclusive**: the bound did not establish robustness, but it does not prove an adversarial example exists. Attack failure in Lab 02 likewise is not a certificate.

The table concerns one image and a few tested radii, rather than test-set accuracy or the exact robust radius. Save your completed code and outputs locally.

Reference: [official auto-LiRPA verification example](https://github.com/Verified-Intelligence/auto_LiRPA/blob/a050a3d61f4cb68b108fa7f07dcb3e4d7ef304df/examples/vision/simple_verification.py). Architecture attribution: [THIRD_PARTY_NOTICES.md](THIRD_PARTY_NOTICES.md).